In [ ]:
# Esto es para construir los modelos
from tensorflow.keras.layers import Dense, Dropout, Input, Flatten, Conv2D ,MaxPooling2D, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import plot_model


# Esto es para entrenamientos
from keras.callbacks import EarlyStopping
from livelossplot import PlotLossesKeras

# Esto es para evaluación

import numpy as np
import pandas as pd
# Graficos mmatrices
import plotly.express as px


In [ ]:
import os
import random
import numpy as np
import tensorflow as tf

SEED = 161105

def set_seeds(seed_value=SEED):
    # 1. Set the PYTHONHASHSEED environment variable at a fixed value
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"
    # 2. Set the python built-in pseudo-random generator at a fixed value
    random.seed(seed_value)
    # 3. Set the numpy pseudo-random generator at a fixed value
    np.random.seed(seed_value)
    # 4. Set the tensorflow pseudo-random generator at a fixed value
    tf.random.set_seed(seed_value)

set_seeds()

In [ ]:
from src import data_loader as dl
from src import EDA
from src import utils
from src import training as trm

# Carga de datos

In [ ]:
X, Y = dl.load_data(fetch = False)

In [ ]:
for tag in X.keys():
    print(X[tag].shape, Y[tag].shape)

# Analisis Exploratorio

In [ ]:
fig1 = EDA.pixels_report(X = X)
fig1.show()

In [ ]:
fig2 = EDA.labels_report(Y = Y)
fig2.show()

## Escalamiento

In [ ]:
X = utils.scaler.apply_scaler(X)   # note the () — you were missing the call}

In [ ]:
fig1 = EDA.pixels_report(X = X)
fig1.show()

In [ ]:
df_imbalance, summary = EDA.imbalance_report(Y, key="train")
display(df_imbalance)
print(summary)

In [ ]:
X_aug, Y_aug = trm.augment_minority_classes(X, Y, key="train", n_minority=3)
# Verify the effect
df_after, summary_after = EDA.imbalance_report(Y_aug, key="train")
display(df_after)
print(summary_after)

In [ ]:
X_aug.keys()

In [ ]:
fig2 = EDA.labels_report(Y = Y_aug)
fig2.show()

In [ ]:
class_weights = trm.compute_class_weights(Y_aug, key="train")
print(class_weights)

In [ ]:
import json
with open('./src/utils/convolutional.json', 'r') as file:
        # Deserialize the file content into a Python object (usually a dictionary or list)
        attributes = json.load(file)
model = trm.skin_classifier(X_aug,Y_aug ,attributes = attributes )


In [ ]:
model.fit(X_aug,Y_aug,'train','test', epochs = 100, use_live_loss = True)

In [ ]:
model.metrics_report(X,Y)

In [ ]:
def export_model(model, path, fmt="keras"):
    """
    Exports a trained Keras model to disk.

    Parameters
    ----------
    model : skin_classifier
        A fitted instance of the classifier.
    path  : str
        Destination path including filename, e.g. "models/my_model".
        Extension is added automatically based on fmt.
    fmt   : str
        "keras" (default, recommended) or "h5" (legacy).
    """
    import os

    valid_formats = ("keras", "h5")
    if fmt not in valid_formats:
        raise ValueError(f"fmt must be one of {valid_formats}, got '{fmt}'")

    if model.history is None:
        raise RuntimeError(
            "Model has not been trained yet. Call .fit() before exporting."
        )

    # Ensure destination directory exists
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    # Strip any extension the user may have included and re-attach the right one
    base = os.path.splitext(path)[0]
    full_path = f"{base}.{fmt}"

    model.model.save(full_path)
    print(f"Model saved to: {full_path}")

    return full_path

In [ ]:
#export_model(model= model, path = './models/dermClass')

In [ ]:
model